In [11]:
# imports y configuración
import warnings; warnings.filterwarnings('ignore')
import time, joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
from interpret.glassbox import ExplainableBoostingClassifier

import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print('Librerías cargadas. Semilla:', RANDOM_SEED)


Librerías cargadas. Semilla: 42


In [12]:
# núcleo de la destilación

def mapa_terminos_single(ebm):
    """feature_idx -> índice de término (solo términos de 1 feature, no interacciones)."""
    return {feats[0]: ti for ti, feats in enumerate(ebm.term_features_) if len(feats) == 1}

def extraer_corte_critico(ebm, term_idx, col_normal):
    """Umbral crítico de una feature = cruce por cero de su shape function.

    La columna `col_normal` de term_scores_ mide cuánto empuja la feature hacia
    la clase Normal. Donde cambia de signo (de >0 'Normal' a <0 'ataque') está
    la frontera que el EBM aprendió. Devuelve ese valor (en espacio escalado).
    """
    edges  = np.asarray(ebm.bins_[term_idx][0], dtype=float)
    scores = ebm.term_scores_[term_idx]
    y = scores[:, col_normal] if scores.ndim == 2 else scores
    x = edges[:-1]
    y = np.asarray(y[:len(x)], dtype=float)
    if len(x) < 2:
        return None
    dif    = np.diff(y)
    cruces = np.where(np.diff(np.sign(y)) != 0)[0]
    if len(cruces) > 0:                       # cruce principal = mayor salto
        k = cruces[np.argmax(np.abs(dif[cruces]))]
    else:                                     # sin cruce: escalón más pronunciado
        k = int(np.argmax(np.abs(dif)))
    return float(x[k])

def destilar_cortes(ebm, n_features, col_normal):
    """Dict feature_idx -> umbral crítico, para todas las features single."""
    mapa = mapa_terminos_single(ebm)
    cortes = {}
    for fi in range(n_features):
        ti = mapa.get(fi)
        if ti is None:
            continue
        thr = extraer_corte_critico(ebm, ti, col_normal)
        if thr is not None:
            cortes[fi] = thr
    return cortes

def construir_binarias(X, cortes, feature_names):
    """Matriz de features binarias (X[:,i] > umbral) + nombres legibles."""
    cols, names = [], []
    for fi, thr in cortes.items():
        cols.append((X[:, fi] > thr).astype(np.float32))
        names.append(f'{feature_names[fi]}>{thr:.3f}')
    return np.column_stack(cols), names

def _eval_bin(y_true, y_pred):
    return {
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_ataque': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision_ataque': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'accuracy': accuracy_score(y_true, y_pred),
    }
print('Funciones de destilación listas.')


Funciones de destilación listas.


In [13]:
# orquestador: destila y compara para un dataset

def destilar_dataset(ebm, X_tr, X_te, y_tr_bin, y_te_bin, y_te_multi,
                     feature_names, normal_label, nombre_dataset, max_depth=4):
    """Destila umbrales del EBM, entrena DT podado sobre features binarias y
    compara contra: (a) EBM como detector binario, (b) DT podado sobre features
    crudas (mismo max_depth)."""
    W = 64
    print('=' * W); print(f' DESTILACIÓN — {nombre_dataset}'.center(W)); print('=' * W)

    clases     = list(ebm.classes_)
    col_normal = clases.index(normal_label)
    n_features = len(feature_names)

    # 1) Umbrales destilados de las shape functions
    cortes = destilar_cortes(ebm, n_features, col_normal)
    print(f'\nUmbrales críticos destilados ({len(cortes)} de {n_features} features):')
    for fi, thr in cortes.items():
        print(f'  {feature_names[fi]:<28} corte (escalado) = {thr:.4f}')

    # 2) Features binarias
    Xb_tr, names_bin = construir_binarias(X_tr, cortes, feature_names)
    Xb_te, _         = construir_binarias(X_te, cortes, feature_names)

    # 3) Alumno: DT podado sobre binarias
    dt_dest = DecisionTreeClassifier(max_depth=max_depth, class_weight='balanced',
                                     random_state=RANDOM_SEED)
    dt_dest.fit(Xb_tr, y_tr_bin)
    m_dest = _eval_bin(y_te_bin, dt_dest.predict(Xb_te))

    # 4a) Baseline: DT podado sobre features CRUDAS (mismo max_depth)
    dt_raw = DecisionTreeClassifier(max_depth=max_depth, class_weight='balanced',
                                    random_state=RANDOM_SEED)
    dt_raw.fit(X_tr, y_tr_bin)
    m_raw = _eval_bin(y_te_bin, dt_raw.predict(X_te))

    # 4b) Maestro: EBM como detector binario (Normal vs no-Normal)
    pred_ebm_bin = (ebm.predict(X_te) != normal_label).astype(int)
    m_ebm = _eval_bin(y_te_bin, pred_ebm_bin)

    # 5) Tabla comparativa
    print(f'\n{"Modelo":<34}{"F1-macro":>10}{"Rec.Atq":>10}{"Prec.Atq":>10}')
    print('-' * W)
    print(f'{"EBM (maestro, binarizado)":<34}{m_ebm["f1_macro"]:>10.4f}'
          f'{m_ebm["recall_ataque"]:>10.4f}{m_ebm["precision_ataque"]:>10.4f}')
    print(f'{"DT podado — features crudas":<34}{m_raw["f1_macro"]:>10.4f}'
          f'{m_raw["recall_ataque"]:>10.4f}{m_raw["precision_ataque"]:>10.4f}')
    print(f'{"DT podado — destiladas (alumno)":<34}{m_dest["f1_macro"]:>10.4f}'
          f'{m_dest["recall_ataque"]:>10.4f}{m_dest["precision_ataque"]:>10.4f}')

    # 6) Reglas del alumno (umbrales con razón teórica del EBM)
    print(f'\nReglas del alumno (DT podado sobre features destiladas):')
    print(export_text(dt_dest, feature_names=names_bin, max_depth=max_depth))

    return {'dataset': nombre_dataset, 'ebm': m_ebm, 'dt_raw': m_raw,
            'dt_dest': m_dest, 'n_cortes': len(cortes), 'cortes': cortes,
            'names_bin': names_bin, 'dt_dest_model': dt_dest}
print('Orquestador listo: destilar_dataset()')


Orquestador listo: destilar_dataset()


In [14]:
from sklearn.linear_model import LogisticRegression

def destilar_contribuciones(ebm, X_tr, X_te, y_tr_bin, y_te_bin,
                            feature_names, normal_label, nombre_dataset, max_depth=4):
    W = 64
    print('=' * W); print(f' DESTILACIÓN-B (contribuciones) — {nombre_dataset}'.center(W)); print('=' * W)

    col_normal = list(ebm.classes_).index(normal_label)
    mapa       = mapa_terminos_single(ebm)
    feat_order = [fi for fi in range(len(feature_names)) if fi in mapa]
    terms      = [mapa[fi] for fi in feat_order]
    names_c    = [f'c[{feature_names[fi]}]' for fi in feat_order]   # contribución de la feature

    def to_contrib(X):
        et = ebm.eval_terms(X)                      # (n, n_terms[, n_clases])
        M  = et[:, terms, col_normal] if et.ndim == 3 else et[:, terms]
        return np.asarray(M, dtype=np.float32)

    Xc_tr, Xc_te = to_contrib(X_tr), to_contrib(X_te)
    print(f'Matriz de contribuciones: {Xc_tr.shape[1]} features (1 por feature single)')

    dt = DecisionTreeClassifier(max_depth=max_depth, class_weight='balanced',
                                random_state=RANDOM_SEED).fit(Xc_tr, y_tr_bin)
    m_dt = _eval_bin(y_te_bin, dt.predict(Xc_te))

    lr = LogisticRegression(max_iter=2000, class_weight='balanced',
                            random_state=RANDOM_SEED).fit(Xc_tr, y_tr_bin)
    m_lr = _eval_bin(y_te_bin, lr.predict(Xc_te))

    m_ebm = _eval_bin(y_te_bin, (ebm.predict(X_te) != normal_label).astype(int))

    print(f'\n{"Modelo":<36}{"F1-macro":>10}{"Rec.Atq":>10}{"Prec.Atq":>10}')
    print('-' * W)
    for nom, m in [('EBM (maestro, binarizado)', m_ebm),
                   ('DT sobre contribuciones (alumno)', m_dt),
                   ('LR sobre contribuciones (alumno)', m_lr)]:
        print(f'{nom:<36}{m["f1_macro"]:>10.4f}{m["recall_ataque"]:>10.4f}{m["precision_ataque"]:>10.4f}')

    print(f'\nReglas del alumno DT (sobre contribuciones EBM):')
    print(export_text(dt, feature_names=names_c, max_depth=max_depth))

    print('Pesos LR (|coef| desc — cuánto pondera la señal-EBM de cada feature):')
    for nom, w in sorted(zip(names_c, lr.coef_[0]), key=lambda t: abs(t[1]), reverse=True):
        print(f'  {nom:<30} {w:+.3f}')

    return {'dataset': nombre_dataset, 'ebm': m_ebm, 'dt': m_dt, 'lr': m_lr}
print('Variante B lista: destilar_contribuciones()')

Variante B lista: destilar_contribuciones()


---
## DATASET 1 — NSL-KDD

In [15]:
# carga (idéntico al pipeline)
COL_NAMES = ['duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty']
FEATURE_COLS_NSL = ['src_bytes','service','dst_bytes','flag','diff_srv_rate',
    'same_srv_rate','dst_host_srv_count','dst_host_same_srv_rate']
DOS={'back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','mailbomb'}
PROBE={'ipsweep','nmap','portsweep','satan','mscan','saint'}
R2L={'ftp_write','guess_passwd','imap','multihop','phf','spy','warezclient','warezmaster','sendmail','named','snmpgetattack','snmpguess','xlock','xsnoop','worm'}
U2R={'buffer_overflow','loadmodule','perl','rootkit','httptunnel','ps','sqlattack','xterm'}
def map_label_nsl(l):
    l=l.lower().strip()
    if l=='normal':return 1
    if l in DOS:return 2
    if l in PROBE:return 3
    if l in R2L:return 4
    if l in U2R:return 5
    return 2
ruta=next((p for p in [(Path.home() / ".cache" / "kagglehub" / "datasets" / "hassan06" / "nslkdd" / "versions" / "1" / "KDDTrain+_20Percent.txt"),Path("KDDTrain+_20Percent.txt"),Path.home()/"Downloads"/"KDDTrain+_20Percent.txt"] if p.exists()),None)
if ruta is None: raise FileNotFoundError("No se encontró KDDTrain+_20Percent.txt")
df=pd.read_csv(ruta,header=None,names=COL_NAMES).drop(columns=['difficulty'])
df['protocol_type']=df['protocol_type'].str.lower().map({'tcp':1,'udp':2,'icmp':3}).fillna(0)
df['service']=LabelEncoder().fit_transform(df['service'].astype(str))
df['flag']=LabelEncoder().fit_transform(df['flag'].astype(str))
y_multi=df['label'].apply(map_label_nsl).values
y_bin=(y_multi>1).astype(int)
X=df[FEATURE_COLS_NSL].values.astype(np.float32)
X_tr_nsl,X_te_nsl,ytr_m_nsl,yte_m_nsl,ytr_b_nsl,yte_b_nsl=train_test_split(
    X,y_multi,y_bin,test_size=0.20,random_state=RANDOM_SEED,stratify=y_multi)
sc=MinMaxScaler(); X_tr_nsl=sc.fit_transform(X_tr_nsl); X_te_nsl=sc.transform(X_te_nsl)
print(f'NSL-KDD: {len(df):,} inst | Train {len(X_tr_nsl):,} | Test {len(X_te_nsl):,}')


NSL-KDD: 25,192 inst | Train 20,153 | Test 5,039


In [16]:
# EBM maestro (carga joblib o entrena)
try:
    ebm_nsl = joblib.load('ebm_nslkdd_stage2.joblib'); print('EBM NSL cargado de joblib.')
except FileNotFoundError:
    print('joblib no encontrado — entrenando EBM multiclase NSL (~160s)...')
    ebm_nsl = ExplainableBoostingClassifier(max_bins=256, interactions=10,
        learning_rate=0.01, max_rounds=5000, min_samples_leaf=2,
        random_state=RANDOM_SEED, feature_names=FEATURE_COLS_NSL).fit(X_tr_nsl, ytr_m_nsl)
    joblib.dump(ebm_nsl, 'ebm_nslkdd_stage2.joblib'); print('EBM NSL entrenado y guardado.')


joblib no encontrado — entrenando EBM multiclase NSL (~160s)...
EBM NSL entrenado y guardado.


In [17]:
res_nsl = destilar_dataset(ebm_nsl, X_tr_nsl, X_te_nsl, ytr_b_nsl, yte_b_nsl,
    yte_m_nsl, FEATURE_COLS_NSL, normal_label=1, nombre_dataset='NSL-KDD')


                      DESTILACIÓN — NSL-KDD                     

Umbrales críticos destilados (8 de 8 features):
  src_bytes                    corte (escalado) = 0.0000
  service                      corte (escalado) = 0.6692
  dst_bytes                    corte (escalado) = 0.0017
  flag                         corte (escalado) = 0.6500
  diff_srv_rate                corte (escalado) = 0.0050
  same_srv_rate                corte (escalado) = 0.0050
  dst_host_srv_count           corte (escalado) = 0.0020
  dst_host_same_srv_rate       corte (escalado) = 0.0050

Modelo                              F1-macro   Rec.Atq  Prec.Atq
----------------------------------------------------------------
EBM (maestro, binarizado)             0.9956    0.9932    0.9974
DT podado — features crudas           0.9747    0.9898    0.9576
DT podado — destiladas (alumno)       0.9373    0.8983    0.9661

Reglas del alumno (DT podado sobre features destiladas):
|--- src_bytes>0.000 <= 0.50
|   |--- diff_srv

In [18]:
res_nsl_B = destilar_contribuciones(ebm_nsl, X_tr_nsl, X_te_nsl, ytr_b_nsl, yte_b_nsl,
    FEATURE_COLS_NSL, normal_label=1, nombre_dataset='NSL-KDD')


            DESTILACIÓN-B (contribuciones) — NSL-KDD            


Matriz de contribuciones: 8 features (1 por feature single)

Modelo                                F1-macro   Rec.Atq  Prec.Atq
----------------------------------------------------------------
EBM (maestro, binarizado)               0.9956    0.9932    0.9974
DT sobre contribuciones (alumno)        0.9840    0.9681    0.9978
LR sobre contribuciones (alumno)        0.9803    0.9770    0.9808

Reglas del alumno DT (sobre contribuciones EBM):
|--- c[src_bytes] <= -0.23
|   |--- c[service] <= 0.26
|   |   |--- c[dst_bytes] <= -0.21
|   |   |   |--- c[dst_bytes] <= -0.70
|   |   |   |   |--- class: 0
|   |   |   |--- c[dst_bytes] >  -0.70
|   |   |   |   |--- class: 1
|   |   |--- c[dst_bytes] >  -0.21
|   |   |   |--- c[service] <= -0.31
|   |   |   |   |--- class: 1
|   |   |   |--- c[service] >  -0.31
|   |   |   |   |--- class: 0
|   |--- c[service] >  0.26
|   |   |--- c[diff_srv_rate] <= 0.24
|   |   |   |--- c[src_bytes] <= -1.16
|   |   |   |   |--- class: 1
|   |   |   |--- c[src_b

---
## DATASET 2 — Mirai

In [19]:
# carga (idéntico al pipeline)
def reclassify_state_mirai(row):
    proto=row.get('state',row.get('proto','OTHER')); b=row.get('b_pkts',0); a=row.get('avg_pkt_size',0)
    if proto=='DNS':return 'DNS_FLOOD'
    if proto in ('HTTP','HTTPS'):return 'HTTP_FLOOD'
    if proto=='SSH':return 'OTHER'
    if proto=='UDP_OTHER':return 'UDP_SMALL_NORESPONSE' if (b==0 and a<100) else ('UDP_LARGE_NORESPONSE' if b==0 else 'UDP_BIDIRECTIONAL')
    if proto=='TCP_OTHER':return 'TCP_SYN_LIKE' if (b==0 and a<80) else ('TCP_ACK_LIKE' if b==0 else 'TCP_ESTABLISHED')
    return 'OTHER'
HWANG={'Normal':76725,'ACK_Flood':7425,'DNS_Flood':4851,'Mirai_CnC':76725,'GREIP_Flood':27801,'HTTP_Flood':135,'SYN_Flood':76725,'UDP_Flood':31878,'VSE_Flood':4986}
FEATURE_COLS_MIRAI=['n_pkts','n_bytes','f_pkts','f_bytes','b_pkts','b_bytes','avg_pkt_size','duration','state']
CLASS_NAMES_MIRAI=list(HWANG.keys())
def find_mirai():
    c=(Path.cwd().parent / "data" / "flows.csv")
    if c.exists():return c
    for base in [Path('.'),Path.home()/'Downloads',Path.home()/'.cache',Path.home()/'markov_mirai']:
        for f in base.rglob('*.csv'):
            if any(k in f.name.lower() for k in ('mirai','hwang','flows')):return f
    raise FileNotFoundError("No se encontró el dataset Mirai.")
try:
    df=pd.read_csv(find_mirai())
    if 'state' in df.columns: df['state']=df.apply(reclassify_state_mirai,axis=1)
    np.random.seed(RANDOM_SEED); keep=[]
    for cls,n in HWANG.items():
        if 'class_name' not in df.columns: continue
        idx=np.where(df['class_name'].values==cls)[0]
        if len(idx)==0: continue
        if len(idx)>n: idx=np.random.choice(idx,n,replace=False)
        keep.append(idx)
    if keep: df=df.iloc[np.sort(np.concatenate(keep))].reset_index(drop=True)
    df['state']=LabelEncoder().fit_transform(df['state'].astype(str))
    c2i={c:i for i,c in enumerate(CLASS_NAMES_MIRAI)}
    y_multi=df['class_name'].map(c2i).values
    y_bin=df['label'].values.astype(int)
    feats_mirai=[f for f in FEATURE_COLS_MIRAI if f in df.columns]
    X=df[feats_mirai].apply(pd.to_numeric,errors='coerce').fillna(0).values.astype(np.float32)
    X_tr_m,X_te_m,ytr_m_m,yte_m_m,ytr_b_m,yte_b_m=train_test_split(X,y_multi,y_bin,test_size=0.20,random_state=RANDOM_SEED,stratify=y_multi)
    sc=MinMaxScaler(); X_tr_m=sc.fit_transform(X_tr_m); X_te_m=sc.transform(X_te_m)
    print(f'Mirai: {len(df):,} inst | feats {len(feats_mirai)} | Train {len(X_tr_m):,} | Test {len(X_te_m):,}')
    MIRAI_OK=True
except FileNotFoundError as e:
    print('AVISO:',e); MIRAI_OK=False


Mirai: 161,384 inst | feats 9 | Train 129,107 | Test 32,277


In [20]:
# EBM maestro + destilación
if MIRAI_OK:
    try:
        ebm_mirai=joblib.load('ebm_mirai_stage2.joblib'); print('EBM Mirai cargado de joblib.')
    except FileNotFoundError:
        print('joblib no encontrado — entrenando EBM multiclase Mirai...')
        ebm_mirai=ExplainableBoostingClassifier(max_bins=256,interactions=10,learning_rate=0.01,
            max_rounds=5000,min_samples_leaf=2,random_state=RANDOM_SEED,feature_names=feats_mirai).fit(X_tr_m,ytr_m_m)
        joblib.dump(ebm_mirai,'ebm_mirai_stage2.joblib'); print('EBM Mirai guardado.')
    res_mirai=destilar_dataset(ebm_mirai,X_tr_m,X_te_m,ytr_b_m,yte_b_m,yte_m_m,
        feats_mirai,normal_label=0,nombre_dataset='Mirai')
else:
    print('Mirai no disponible — bloque saltado.')


joblib no encontrado — entrenando EBM multiclase Mirai...
EBM Mirai guardado.
                       DESTILACIÓN — Mirai                      

Umbrales críticos destilados (9 de 9 features):
  n_pkts                       corte (escalado) = 0.0002
  n_bytes                      corte (escalado) = 0.0000
  f_pkts                       corte (escalado) = 0.0000
  f_bytes                      corte (escalado) = 0.0000
  b_pkts                       corte (escalado) = 0.0029
  b_bytes                      corte (escalado) = 0.0000
  avg_pkt_size                 corte (escalado) = 0.0085
  duration                     corte (escalado) = 0.0000
  state                        corte (escalado) = 0.3125

Modelo                              F1-macro   Rec.Atq  Prec.Atq
----------------------------------------------------------------
EBM (maestro, binarizado)             0.9885    0.9981    0.9996
DT podado — features crudas           0.9727    0.9946    0.9999
DT podado — destiladas (alumno)   

In [21]:
if MIRAI_OK:
    res_mirai_B = destilar_contribuciones(ebm_mirai, X_tr_m, X_te_m, ytr_b_m, yte_b_m,
        feats_mirai, normal_label=0, nombre_dataset='Mirai')
else:
    print('Mirai no disponible — bloque saltado.')


             DESTILACIÓN-B (contribuciones) — Mirai             
Matriz de contribuciones: 9 features (1 por feature single)

Modelo                                F1-macro   Rec.Atq  Prec.Atq
----------------------------------------------------------------
EBM (maestro, binarizado)               0.9885    0.9981    0.9996
DT sobre contribuciones (alumno)        0.9844    0.9972    0.9997
LR sobre contribuciones (alumno)        0.9841    0.9970    0.9998

Reglas del alumno DT (sobre contribuciones EBM):
|--- c[avg_pkt_size] <= 1.50
|   |--- c[avg_pkt_size] <= 1.15
|   |   |--- c[duration] <= 1.69
|   |   |   |--- c[n_bytes] <= 2.98
|   |   |   |   |--- class: 1
|   |   |   |--- c[n_bytes] >  2.98
|   |   |   |   |--- class: 0
|   |   |--- c[duration] >  1.69
|   |   |   |--- c[f_pkts] <= -1.34
|   |   |   |   |--- class: 1
|   |   |   |--- c[f_pkts] >  -1.34
|   |   |   |   |--- class: 0
|   |--- c[avg_pkt_size] >  1.15
|   |   |--- c[state] <= 0.60
|   |   |   |--- c[avg_pkt_size] <= 

---
## DATASET 3 — DS2OS

In [22]:
# carga (idéntico al pipeline)
CLASS_MAP={'normal':'Normal','anomalous(DoSattack)':'DoS','anomalous(scan)':'Scan',
 'anomalous(malitiousControl)':'MaliciousControl','anomalous(malitiousOperation)':'MaliciousOperation',
 'anomalous(spying)':'Spying','anomalous(dataProbing)':'DataProbing','anomalous(wrongSetUp)':'WrongSetUp'}
CLASS_TO_INT={v:i for i,v in enumerate(CLASS_MAP.values())}
CAT_COLS=['sourceID','sourceAddress','sourceType','sourceLocation','destinationServiceAddress',
 'destinationServiceType','destinationLocation','accessedNodeAddress','accessedNodeType','operation']
FEAT_DS2OS=CAT_COLS+['value']
try:
    ruta=next((p for p in [Path("DS2OS.csv"),Path.home()/"Downloads"/"DS2OS.csv",
        (Path.home() / ".cache" / "kagglehub" / "datasets" / "libamariyam" / "ds2os-dataset" / "versions" / "1" / "DS2OS.csv")] if p.exists()),None)
    if ruta is None: raise FileNotFoundError("No se encontró DS2OS.csv")
    df=pd.read_csv(ruta)
    df['accessedNodeType']=df['accessedNodeType'].fillna('Malicious')
    df['value']=df['value'].replace({'False':0,'True':1,'Twenty':20,'none':0})
    df['value']=pd.to_numeric(df['value'],errors='coerce').fillna(0)
    df=df.drop(columns=['timestamp'])
    df['y']=df['normality'].map(CLASS_MAP).map(CLASS_TO_INT).astype(np.int8)
    y_bin=(df['normality']!='normal').astype(int).values
    df=df.drop(columns=['normality'])
    for col in CAT_COLS: df[col]=LabelEncoder().fit_transform(df[col].astype(str))
    X=df[FEAT_DS2OS].values.astype(np.float32); y_multi=df['y'].values
    X_tr_d,X_te_d,ytr_m_d,yte_m_d,ytr_b_d,yte_b_d=train_test_split(X,y_multi,y_bin,test_size=0.20,random_state=RANDOM_SEED,stratify=y_multi)
    sc=StandardScaler(); X_tr_d=sc.fit_transform(X_tr_d); X_te_d=sc.transform(X_te_d)
    print(f'DS2OS: {len(df):,} inst | feats {len(FEAT_DS2OS)} | Train {len(X_tr_d):,} | Test {len(X_te_d):,}')
    DS2OS_OK=True
except FileNotFoundError as e:
    print('AVISO:',e); DS2OS_OK=False


DS2OS: 357,952 inst | feats 11 | Train 286,361 | Test 71,591


In [23]:
# EBM maestro + destilación
if DS2OS_OK:
    try:
        ebm_ds2os=joblib.load('ebm_ds2os_stage2.joblib'); print('EBM DS2OS cargado de joblib.')
    except FileNotFoundError:
        print('joblib no encontrado — entrenando EBM multiclase DS2OS...')
        ebm_ds2os=ExplainableBoostingClassifier(max_bins=256,interactions=10,learning_rate=0.01,
            max_rounds=5000,min_samples_leaf=2,n_jobs=1,random_state=RANDOM_SEED,feature_names=FEAT_DS2OS).fit(X_tr_d,ytr_m_d)
        joblib.dump(ebm_ds2os,'ebm_ds2os_stage2.joblib'); print('EBM DS2OS guardado.')
    res_ds2os=destilar_dataset(ebm_ds2os,X_tr_d,X_te_d,ytr_b_d,yte_b_d,yte_m_d,
        FEAT_DS2OS,normal_label=0,nombre_dataset='DS2OS')
else:
    print('DS2OS no disponible — bloque saltado.')


joblib no encontrado — entrenando EBM multiclase DS2OS...
EBM DS2OS guardado.
                       DESTILACIÓN — DS2OS                      

Umbrales críticos destilados (11 de 11 features):
  sourceID                     corte (escalado) = -1.2567
  sourceAddress                corte (escalado) = 0.9164
  sourceType                   corte (escalado) = 0.3329
  sourceLocation               corte (escalado) = -1.4687
  destinationServiceAddress    corte (escalado) = -1.5781
  destinationServiceType       corte (escalado) = -0.4433
  destinationLocation          corte (escalado) = 1.4869
  accessedNodeAddress          corte (escalado) = -1.5988
  accessedNodeType             corte (escalado) = -0.6804
  operation                    corte (escalado) = -1.0259
  value                        corte (escalado) = 24.9575

Modelo                              F1-macro   Rec.Atq  Prec.Atq
----------------------------------------------------------------
EBM (maestro, binarizado)             0.

In [24]:
if DS2OS_OK:
    res_ds2os_B = destilar_contribuciones(ebm_ds2os, X_tr_d, X_te_d, ytr_b_d, yte_b_d,
        FEAT_DS2OS, normal_label=0, nombre_dataset='DS2OS')
else:
    print('DS2OS no disponible — bloque saltado.')


             DESTILACIÓN-B (contribuciones) — DS2OS             
Matriz de contribuciones: 11 features (1 por feature single)

Modelo                                F1-macro   Rec.Atq  Prec.Atq
----------------------------------------------------------------
EBM (maestro, binarizado)               0.9389    0.7978    0.9834
DT sobre contribuciones (alumno)        0.8632    0.8442    0.6511
LR sobre contribuciones (alumno)        0.7603    0.9141    0.3864

Reglas del alumno DT (sobre contribuciones EBM):
|--- c[accessedNodeAddress] <= -0.04
|   |--- c[sourceAddress] <= 0.17
|   |   |--- c[sourceLocation] <= -0.53
|   |   |   |--- c[sourceID] <= 0.29
|   |   |   |   |--- class: 1
|   |   |   |--- c[sourceID] >  0.29
|   |   |   |   |--- class: 0
|   |   |--- c[sourceLocation] >  -0.53
|   |   |   |--- c[destinationLocation] <= -0.17
|   |   |   |   |--- class: 1
|   |   |   |--- c[destinationLocation] >  -0.17
|   |   |   |   |--- class: 1
|   |--- c[sourceAddress] >  0.17
|   |   |--- 

---
## COMPARATIVA FINAL — Destilación

In [25]:
# tabla resumen (F1-macro binario)
filas=[(res_nsl,res_nsl_B)]
if MIRAI_OK: filas.append((res_mirai,res_mirai_B))
if DS2OS_OK: filas.append((res_ds2os,res_ds2os_B))

W=92
print('='*W); print(' DESTILACIÓN DE EXPLICACIONES — RESUMEN (F1-macro binario)'.center(W)); print('='*W)
print(f'{"Dataset":<10}{"EBM":>9}{"DT crudo":>10}{"A: single":>12}{"B: contrib-DT":>15}{"B: contrib-LR":>15}')
print('-'*W)
for rA,rB in filas:
    print(f'{rA["dataset"]:<10}{rA["ebm"]["f1_macro"]:>9.4f}{rA["dt_raw"]["f1_macro"]:>10.4f}'
          f'{rA["dt_dest"]["f1_macro"]:>12.4f}{rB["dt"]["f1_macro"]:>15.4f}{rB["lr"]["f1_macro"]:>15.4f}')
print('-'*W)



                  DESTILACIÓN DE EXPLICACIONES — RESUMEN (F1-macro binario)                 
Dataset         EBM  DT crudo   A: single  B: contrib-DT  B: contrib-LR
--------------------------------------------------------------------------------------------
NSL-KDD      0.9956    0.9747      0.9373         0.9840         0.9803
Mirai        0.9885    0.9727      0.7892         0.9844         0.9841
DS2OS        0.9389    0.8407      0.3615         0.8632         0.7603
--------------------------------------------------------------------------------------------
